# factorscope on real Fama-French data

A statistical risk model runs PCA on returns and calls the eigenvectors "factors" — but
PCA only recovers the *subspace*; the factors themselves are scrambled by an arbitrary
rotation. This notebook shows, on **real Fama-French portfolios**, how `factorscope`:

1. detects when the rotation is untrustworthy and fixes it (removing the market),
2. **labels** the recovered factors against known style factors,
3. beats plain PCA at recovering the named factors — *in the regime where that's possible*,
4. produces factors that **persist across refits**.

It is deliberately honest about the limits: on a raw, market-dominated panel SOBI does
**not** help, and `factorscope` says so rather than returning silent garbage.

In [1]:
import numpy as np
import pandas as pd

from factorscope import FactorModel, load_portfolios, load_reference_factors
from factorscope.alignment import matched_corr

port = load_portfolios("100_size_bm", start="1970-01-01")
ff = load_reference_factors("ff5", start="1970-01-01")
idx = port.index.intersection(ff.index)
port, ff = port.loc[idx], ff.loc[idx]
print(f"{port.shape[1]} portfolios x {len(port)} days,  {idx[0].date()} .. {idx[-1].date()}")

100 portfolios x 14223 days,  1970-01-02 .. 2026-05-29


## 1. The problem: is the rotation even identifiable?

On raw equity returns the **market** dominates the spectrum (one component owns ~70% of the
variance). When one eigenvalue is that large, the PCA rotation is *already pinned* — there is
no ambiguity for a time-structure rotation to resolve, and rotating only injects noise.

`factorscope`'s guardrail detects this and refuses to certify the labels.

In [2]:
fm_raw = FactorModel(n_factors=5, neutralize=False).fit(port)
print(fm_raw.trust_report())

TrustReport
  regime            : dominated
  top-PC share      : 0.69
  market neutralized: False
  identif. margin   : 9.39  (OK)
  verdict           : Do NOT trust factor labels as-is (subspace is fine; orientation is not).
  ! Spectrum dominated by one factor: PCA is already identified; rotation may hurt. Neutralize the market first.


## 2. The fix: remove the market, then rotate the residual

`neutralize="auto"` detects the dominated regime, cross-sectionally demeans each day to strip
the market, and rotates the (now flat-spectrum) residual — the regime where SOBI is
identifiable. The verdict flips to *trustworthy*.

In [3]:
fm = FactorModel(n_factors=5, neutralize="auto").fit(port)
print("auto-neutralized:", fm.neutralized_)
print(fm.trust_report())

auto-neutralized: True
TrustReport
  regime            : flat
  top-PC share      : 0.15
  market neutralized: True
  identif. margin   : 10.11  (OK)
  verdict           : Factors are identifiable and separated -- labels can be trusted.


## 3. Label the factors (Feature 1)

Regress each recovered factor on the Fama-French factors to give it a human name. With the
market removed, the residual factors line up with the **style** factors (size, value, ...).

In [4]:
lab = fm.label(reference=ff)
print(lab)
lab.table.round(2)

LabelResult
  F1     ~ +0.21*HML +0.18*Mkt-RF       (R2=0.12)
  F2     ~ +0.56*HML -0.41*SMB          (R2=0.65)
  F3     ~ -0.62*HML -0.40*SMB          (R2=0.61)
  F4     ~ +0.40*Mkt-RF -0.32*HML       (R2=0.47)
  F5     ~ +0.70*SMB -0.21*HML          (R2=0.45)


,Mkt-RF,SMB,HML,RMW,CMA
F1,0.18,-0.17,0.21,-0.16,-0.15
F2,0.06,-0.41,0.56,0.29,0.10
F3,-0.29,-0.40,-0.62,0.08,0.10
F4,0.40,-0.17,-0.32,-0.02,-0.14
F5,0.07,0.70,-0.21,0.10,0.11


## 4. Does SOBI actually beat PCA here?

Match the recovered factors to the four non-market FF5 factors (SMB, HML, RMW, CMA) with a
permutation-invariant score. SOBI's rotation should recover them better than the raw PCA
eigenportfolios of the residual.

In [5]:
target = ff[["SMB", "HML", "RMW", "CMA"]].values
fm_pca = FactorModel(n_factors=5, neutralize=True, rotation="none").fit(port)
print(f"PCA  matched |corr|: {matched_corr(fm_pca.factors_.values, target):.3f}")
print(f"SOBI matched |corr|: {matched_corr(fm.factors_.values, target):.3f}")

PCA  matched |corr|: 0.488
SOBI matched |corr|: 0.533


The SOBI edge is **real but modest** (~+0.04–0.08). That is the honest scale of the
effect — not the "99% recovery" sometimes claimed. The differentiated value of the tool is
not a huge rotation win; it is *knowing which regime you're in* and labelling/monitoring the
factors reliably.

## 5. Do the factors persist across refits? (Feature 4)

A risk desk refits periodically; PCA factor identity churns when eigenvalues are close.
SOBI's rotation is pinned by time structure, so its factors should be more stable.

In [6]:
st = fm.stability_report(window=1000, step=250)
print(st)

StabilityResult
  refits            : 52  (window=1000, step=250)
  SOBI persistence  : 0.816
  PCA  persistence  : 0.803


## The honest summary

- **Raw market-dominated panel:** SOBI loses to PCA — and `trust_report()` warns you.
- **Market-neutral panel with real style structure:** SOBI wins and the labels are trustworthy.
- **A panel without the factors** (e.g. industry portfolios): everything sits at chance —
  the tool recovers *nothing* rather than hallucinating factors that aren't there.

Knowing which of these three situations you are in — before you trust a factor label — is
what `factorscope` is for.